# DATASET_INSPECTOR.ipynb

Inspect REVIDE folder structure before training TRDN.


## Install / Imports


In [ ]:
import sys, subprocess
try:
    import torch, PIL, matplotlib
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "Pillow", "matplotlib", "torch", "torchvision", "opencv-python"])
from pathlib import Path
import sys, json
import matplotlib.pyplot as plt
from PIL import Image
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").exists(): sys.path.insert(0, str(REPO_ROOT))
if (REPO_ROOT.parent / "src").exists(): sys.path.insert(0, str(REPO_ROOT.parent))
from src.config import TRDNConfig
from src.dataset import discover_revide_sequences, image_to_tensor
cfg = TRDNConfig()
print(json.dumps(cfg.to_dict(), indent=2, default=str))


## Mount Google Drive


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', repr(exc))


## Scan REVIDE Folders


In [ ]:
def inspect_root(root, split):
    root = Path(root)
    sequences = discover_revide_sequences(root, split=None, extensions=cfg.image_extensions)
    print(f"{split}: root={root} exists={root.exists()} sequences={len(sequences)}")
    total_hazy = sum(len(s['hazy_files']) for s in sequences)
    total_clean = sum(len(s['clean_files']) for s in sequences)
    print(f"  hazy frames={total_hazy} clean frames={total_clean}")
    for seq in sequences[:5]:
        print(f"  {seq['name']}: hazy={len(seq['hazy_files'])} clean={len(seq['clean_files'])}")
    return sequences
train_sequences = inspect_root(cfg.train_root, 'train')
test_sequences = inspect_root(cfg.test_root, 'test')


## Inspect Image Sizes


In [ ]:
from collections import Counter

def size_counter(sequences, max_images=100):
    counter = Counter()
    for seq in sequences:
        for path in (seq['hazy_files'][:max_images] + seq['clean_files'][:max_images]):
            try:
                with Image.open(path) as img:
                    counter[img.size] += 1
            except Exception as exc:
                print('failed', path, exc)
    return counter
print('train sizes:', size_counter(train_sequences).most_common(10))
print('test sizes:', size_counter(test_sequences).most_common(10))


## Display Samples


In [ ]:
def show_sample(sequences, title):
    if not sequences:
        print('No sequences available for', title)
        return
    seq = sequences[0]
    hazy = image_to_tensor(seq['hazy_files'][0])
    clean = image_to_tensor(seq['clean_files'][0])
    plt.figure(figsize=(10,4))
    for i,(img,name) in enumerate([(hazy,'hazy'),(clean,'clean')],1):
        plt.subplot(1,2,i)
        plt.imshow(img.permute(1,2,0).clamp(0,1))
        plt.title(f"{title} {seq['name']} {name}")
        plt.axis('off')
    plt.tight_layout(); plt.show()
show_sample(train_sequences, 'train')
show_sample(test_sequences, 'test')


## Path Structure Verdict


In [ ]:
issues=[]
if not Path(cfg.train_root).exists(): issues.append(f"Missing train_root: {cfg.train_root}")
if not Path(cfg.test_root).exists(): issues.append(f"Missing test_root: {cfg.test_root}")
if not train_sequences: issues.append('No train sequences discovered')
if not test_sequences: issues.append('No test sequences discovered')
if issues:
    print('Issues found:')
    for issue in issues: print(' -', issue)
else:
    print('Dataset structure looks usable for TRDN training/validation.')
